# 00 환경과 모델 입력 계약

무엇을 하는가: 서버 경로, GPU, 공식 checkpoint를 확인한다. 설치 성공과 HCC 적용 실험은 별도 상태다. 이 실습은 이미 내려받은 작은 입력을 사용하며 다운로드를 시작하지 않는다.

In [1]:
from pathlib import Path
import os, sys, json, subprocess
ROOT = Path.cwd()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
assert (ROOT / 'configs/paths.yaml').exists()
os.chdir(ROOT)
os.environ.update(OMP_NUM_THREADS='8', OPENBLAS_NUM_THREADS='8', MKL_NUM_THREADS='8', TOKENIZERS_PARALLELISM='false')
os.environ.setdefault('CUDA_VISIBLE_DEVICES', '0')
import yaml, pandas as pd, numpy as np
PATHS = yaml.safe_load((ROOT / 'configs/paths.yaml').read_text())
DATA = Path(PATHS['data_root'])
def run_script(name, config=None, model='scgpt', args=()):
    cmd = [str(ROOT / f'.venvs/vcell-{model}/bin/python'), '-u', str(ROOT / 'scripts' / name)]
    if config: cmd += ['--config', str(ROOT / 'configs' / config)]
    cmd += list(args)
    print(' '.join(cmd))
    p = subprocess.run(cmd, cwd=ROOT, capture_output=True, text=True)
    print(p.stdout[-16000:])
    if p.returncode:
        print(p.stderr[-16000:])
        p.check_returncode()
    return p
print('code:', ROOT, '\ndata:', DATA)


code: /mnt/S1/sdata/workbench/Virtual_Cell/gssong/vcell_liver 
data: /mnt/S1/sdata/pubdata/Virtual_Cell/gssong/vcell_liver


In [2]:
run_script('00_inspect_environment.py', args=['--out', 'reports/environment_notebook.json'])
print(json.loads((ROOT / 'reports/environment_notebook.json').read_text())['torch_cuda'])

/mnt/S1/sdata/workbench/Virtual_Cell/gssong/vcell_liver/.venvs/vcell-scgpt/bin/python -u /mnt/S1/sdata/workbench/Virtual_Cell/gssong/vcell_liver/scripts/00_inspect_environment.py --out reports/environment_notebook.json


{
  "hostname": "G1",
  "platform": "Linux-5.15.0-124-generic-x86_64-with-glibc2.35",
  "python": "3.10.10 | packaged by conda-forge | (main, Mar 24 2023, 20:08:06) [GCC 11.3.0]",
  "executable": "/mnt/S1/sdata/workbench/Virtual_Cell/gssong/vcell_liver/.venvs/vcell-scgpt/bin/python",
  "cpu": "x86_64",
  "logical_cpu_count": 80,
  "ram_bytes": 1081788682240,
  "disk": {
    "total": 941231351791616,
    "used": 698006340894720,
    "free": 243225010896896
  },
  "torch": "2.5.1+cu124",
  "torch_cuda": "12.4",
  "cuda_available": true,
  "gpu": {
    "command": [
      "nvidia-smi"
    ],
    "exit_code": 0,
    "output": "Mon Oct 26 14:16:27 2026       \n+-----------------------------------------------------------------------------------------+\n| NVIDIA-SMI 560.28.03              Driver Version: 560.28.03      CUDA Version: 12.6     |\n|-----------------------------------------+------------------------+----------------------+\n| GPU  Name                 Persistence-M | Bus-Id        

In [3]:
print(json.loads((DATA / 'checkpoints/scgpt-whole-human/args.json').read_text()))
print(json.loads((ROOT / 'third_party/Geneformer/Geneformer-V2-104M/config.json').read_text()))
print(pd.read_csv(ROOT / 'reports/patient_label_counts.tsv', sep='\t').to_string(index=False))

{'data_source': '/scratch/ssd004/datasets/cellxgene/scb_strict/human', 'save_dir': '/scratch/ssd004/datasets/cellxgene/save/cellxgene_census_human-May23-08-36-2023', 'load_model': None, 'n_hvg': None, 'valid_size_or_ratio': 0.003, 'dist_backend': 'nccl', 'grad_accu_steps': 1, 'pad_token': '<pad>', 'input_style': 'binned', 'input_emb_style': 'continuous', 'n_bins': 51, 'max_seq_len': 1200, 'training_tasks': 'both', 'dist_url': 'tcp://gpu188.cluster.local:53833', 'mask_ratio': [0.25, 0.5, 0.75], 'trunc_by_sample': True, 'vocab_path': '/scratch/ssd004/datasets/cellxgene/scFormer/scformer/tokenizer/default_census_vocab.json', 'rank': 0, 'batch_size': 32, 'eval_batch_size': 64, 'epochs': 6, 'lr': 0.0001, 'scheduler_interval': 100, 'scheduler_factor': 0.99, 'warmup_ratio_or_step': 10000.0, 'no_cls': True, 'no_cce': True, 'fp16': True, 'fast_transformer': True, 'nlayers': 12, 'nheads': 8, 'embsize': 512, 'd_hid': 512, 'dropout': 0.2, 'n_layers_cls': 3, 'log_interval': 9000, 'save_interval': 2

생물학적 해석: scGPT는 공식 vocab의 발현 rank bins, Geneformer V2는 전체 raw count·Ensembl·n_counts로 median-normalized rank encoding을 만든다. PCA는 train 환자에서만 학습한다. frozen embedding + supervised head이며 분류 시스템 전체를 zero-shot이라 부르지 않는다.

실패 해결: 경로는 configs/paths.yaml과 envs/activate.sh 확인. CUDA가 없으면 scGPT는 CPU로 동작하고 Geneformer도 느린 CPU fallback이 있다. 다른 GPU 작업을 종료하지 않는다. GEARS의 실측 개입 예측은 별도 task다.